# Q33B — ARA-first boundary-child flow route

## tl;dr

Q33B keeps the ARA geometry fixed:

`2 + (1 + boundary child projected 1→0.5) = 3.5`.

It does not estimate `0.5` from energy. Instead, the geometry selects the one
endpoint child nearest the low boundary and predicts that relation closure
will rise after the high-side source releases.

Across 11,543 evaluation events, exact boundary flow was positive in 63.64%,
versus 55.83% for the sibling and 50.79–56.38% for controls. Every
trial-cluster bootstrap comparison gave probability 1.000.

Frozen verdict: **BOUNDARY-CHILD FLOW ROUTE SUPPORTED INSIDE THIS SIMULATOR**

Independent validation: **PASS**.


## Context & Methods

### Key assumptions

- ARA supplies invariant geometry; measured closure and energy are variable
  flows over it.
- The child singularity and parent ridge are the same adjacent-rung boundary.
- A complete boundary child projects from `1` locally to `0.5` in the parent.
- The source releases from high to low, so the endpoint child with smaller
  starting normalized closure is the directed recipient.

For each endpoint relation:

\[
z_c(t)=\frac{h_c(t)}{Q_{.95}^{dev}(h_c)},\qquad
g_c(t)=\frac{h_c(t+1)-h_c(t)}{Q_{.95}^{dev}(h_c)},
\quad h=|\det C|^{1/3}.
\]

Starting `z` selects the route without future values. Next `g` is the scored
flow. Sibling, topology, seed and time controls all apply the same lower-of-two
rule.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Q33B_ARA_FIRST_BOUNDARY_CHILD_RESULTS.json").exists():
    ROOT = Path(r"F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\quantum")

result = json.loads((ROOT / "Q33B_ARA_FIRST_BOUNDARY_CHILD_RESULTS.json").read_text())
validation = json.loads((ROOT / "Q33B_ARA_FIRST_BOUNDARY_CHILD_VALIDATION.json").read_text())
evaluation = result["splits"]["evaluation"]
print("Structural path:", result["geometry"]["complete_path"])
print("Evaluation events:", evaluation["source_events"])
print("Verdict:", result["frozen_verdict"]["label"])
print("Independent validation:", validation["status"])


Structural path: 3.5
Evaluation events: 11543
Verdict: BOUNDARY-CHILD FLOW ROUTE SUPPORTED INSIDE THIS SIMULATOR
Independent validation: PASS


## Results

### Route flow


In [2]:
routes = pd.DataFrame([
    {
        "route": route,
        "events": evaluation["routes"][route]["paired_events"],
        "median flow": evaluation["routes"][route]["flow"]["median"],
        "mean flow": evaluation["routes"][route]["flow"]["mean"],
        "positive fraction": evaluation["routes"][route]["positive_fraction"],
        "median starting z": evaluation["routes"][route]["start_z"]["median"],
    }
    for route in ("exact", "sibling", "topology", "seed", "time")
])
routes


route,events,median flow,mean flow,positive fraction,median starting z
exact,11543,0.041425,0.049524,0.636403,0.138506
sibling,11543,0.040806,0.025701,0.558260,0.630249
topology,11543,0.000331,0.013352,0.507927,0.057304
seed,10788,0.002365,0.021457,0.563775,0.038448
time,10790,0.002034,0.021258,0.560241,0.040841


The exact route is more reliably positive. The sibling has a
similar marginal median but a wider distribution, lower mean and lower
positive fraction.


### Paired comparisons


In [3]:
paired = pd.DataFrame([
    {
        "comparator": comparator,
        "median exact minus comparator": evaluation["paired_differences"][comparator]["median"],
        "cluster mean difference": result["evaluation_bootstrap"][comparator]["mean_exact_minus_comparator"],
        "bootstrap P(exact greater)": result["evaluation_bootstrap"][comparator]["probability_exact_greater"],
    }
    for comparator in ("sibling", "topology", "seed", "time")
])
paired


comparator,median exact minus comparator,cluster mean difference,bootstrap P(exact greater)
sibling,0.017813,0.023629,1.0
topology,0.033847,0.036031,1.0
seed,0.029968,0.027917,1.0
time,0.029093,0.028325,1.0


### Branch replication


In [4]:
branches = pd.DataFrame([
    {
        "branch": branch,
        "events": evaluation["branches"][branch]["source_events"],
        "median exact flow": evaluation["branches"][branch]["exact_flow"]["median"],
        "positive fraction": evaluation["branches"][branch]["exact_positive_fraction"],
    }
    for branch in ("c2", "c4")
])
branches


branch,events,median exact flow,positive fraction
c2,5772,0.042808,0.633403
c4,5771,0.039643,0.639404


### Geometry

![Q33B geometry](Q33B_ARA_FIRST_BOUNDARY_CHILD_GEOMETRY.png)

All frozen routing gates passed. Controls confirm some generic lower-of-two
mean reversion, but the endpoint-specific boundary route retains an additional
7.26–12.85 percentage-point positive-flow advantage.


## Takeaways

1. Keeping `0.5` structural rather than estimating it corrected Q33's
   coordinate error.
2. The fixed route successfully selects the more reliable closure-flow
   recipient inside this simulator.
3. The result is stable across `c2`, `c4`, development and evaluation.
4. It supports the directed boundary-child consequence, not a numerical
   derivation or independent test of `3.5`.
5. Raw energy flow does not show the same clean sibling ordering; the supported
   observable is relation closure.

## Caveats

This source is already-open, simulated and exactly diagonal. Lower-of-two
selection creates generic headroom, so same-rule controls are essential.
Starting distributions are not perfectly identical. This is not hardware
quantum evidence, universal ARA, Phase B or a dark-sector validation.
